# PCA and the shared response model
Fit 64-dimensional PCA on training-question vectors only. Jointly learn a shared scorer and eight numbers per background respondent. Validation/test answers and development/final respondent evidence stay unread. Metrics below measure training fit only.


In [1]:
import hashlib
import json
import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.decomposition import PCA
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
OUT = ROOT / "outputs"
def read_json(name):
    return json.loads((ROOT / name).read_text())
def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

config = read_json("configs/response_model.json")
split = read_json("data/splits/question_split.json")
people = read_json("data/splits/respondent_split.json")["groups"]
embedding_audit = read_json("outputs/question_answer_embedding_audit.json")
index = pd.DataFrame(read_json("outputs/question_answer_index.json"))
embeddings = np.load(OUT / "question_answer_embeddings.npy", allow_pickle=False)
for path, spec in embedding_audit["files"].items():
    assert sha256(ROOT / path) == spec["sha256"]
assert sha256(ROOT / "configs/embedding_model.json") == embedding_audit["input_sha256"]["configs/embedding_model.json"]
assert embeddings.shape == (1877, 1024) and np.isfinite(embeddings).all()
assert index["embedding_row"].tolist() == list(range(len(embeddings)))
assert not index.duplicated(["item_id", "answer_index"]).any()
inputs = ["configs/response_model.json", "data/splits/question_split.json", "data/splits/respondent_split.json",
          "outputs/question_answer_embeddings.npy", "outputs/question_answer_index.json", config["training_partition"]]
input_hashes = {p: sha256(ROOT / p) for p in inputs}
np.random.seed(config["seed"])
torch.manual_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Training device: {device}")


Training device: mps


## 1. Fit PCA on training options
Whitening scales the 64 retained coordinates to comparable training variance. Save this fitted transformation for all later questions; never refit it on validation or test inputs.


In [2]:
training_questions = split["training_columns"]
training_rows = np.flatnonzero(index["item_id"].isin(training_questions).to_numpy())
assert len(training_rows) == 1822
assert set(index.iloc[training_rows]["question_role"]) == {"training"}
assert set(index.iloc[training_rows]["item_id"]) == set(training_questions)
assert not set(training_questions) & set(split["forbidden_input_columns_all_respondents_all_waves"])

start = time.monotonic()
pca = PCA(n_components=config["pca_components"], whiten=config["pca_whiten"],
          svd_solver=config["pca_solver"], random_state=config["seed"])
pca.fit(embeddings[training_rows])
features = pca.transform(embeddings).astype(np.float32)
pca_seconds = time.monotonic() - start
assert pca.n_samples_ == len(training_rows)
assert features.shape == (1877, 64) and np.isfinite(features).all()
assert np.all(pca.explained_variance_ > 0)
manual = ((embeddings - pca.mean_) @ pca.components_.T) / np.sqrt(pca.explained_variance_)
np.testing.assert_allclose(features, manual, atol=2e-5, rtol=2e-4)
for _, rows in index.groupby("item_id", sort=False):
    values = features[rows["embedding_row"].to_numpy()]
    assert len(np.unique(values, axis=0)) == len(values)

# Overall variance can hide loss of differences between a question's options.
within_before, within_after = 0.0, 0.0
for _, rows in index.iloc[training_rows].groupby("item_id", sort=False):
    values = embeddings[rows["embedding_row"].to_numpy()]
    differences = values - values.mean(axis=0)
    within_before += float(np.square(differences).sum())
    within_after += float(np.square(differences @ pca.components_.T).sum())
option_variance_retained = within_after / within_before

pca_path = OUT / "question_answer_pca.npz"
feature_path = OUT / "question_answer_features_64.npy"
np.savez(pca_path, mean=pca.mean_, components=pca.components_,
         explained_variance=pca.explained_variance_, explained_variance_ratio=pca.explained_variance_ratio_,
         training_embedding_rows=training_rows, whiten=np.array(config["pca_whiten"]))
np.save(feature_path, features, allow_pickle=False)
print(f"PCA: {features.shape}; {pca.explained_variance_ratio_.sum():.1%} training variance retained; {pca_seconds:.2f}s")


PCA: (1877, 64); 85.7% training variance retained; 0.02s


## 2. Prepare training tensors
Every batch contains people and their 430 questions. Missing answers use `-1`; padded options are masked out. Feature rows retain the existing question/option index.


In [3]:
wide = pd.read_parquet(ROOT / config["training_partition"])
manifest = read_json("data/splits/partition_manifest.json")
partition = next(p for p in manifest["partitions"] if p["partition"] == "shared_training")
assert sha256(ROOT / config["training_partition"]) == partition["sha256"]
assert wide["pid"].tolist() == people["shared_training"] and wide["pid"].is_unique
assert wide.columns.tolist() == ["pid"] + training_questions
assert not set(wide["pid"]) & (set(people["development"]) | set(people["final_personas"]))
person_ids = wide["pid"].tolist()

option_counts = index.groupby("item_id").size()
max_options = int(option_counts.loc[training_questions].max())
E = np.zeros((len(training_questions), max_options, 64), dtype=np.float32)
option_mask = np.zeros((len(training_questions), max_options), dtype=bool)
for q, item_id in enumerate(training_questions):
    rows = index.loc[index["item_id"].eq(item_id)].sort_values("answer_index")
    assert rows["answer_index"].tolist() == list(range(len(rows)))
    E[q, :len(rows)] = features[rows["embedding_row"].to_numpy()]
    option_mask[q, :len(rows)] = True
Y = wide[training_questions].to_numpy(dtype=np.float64, na_value=np.nan)
Y = np.where(np.isnan(Y), -1, Y - 1).astype(np.int64)
observed = Y >= 0
assert int((~observed).sum()) == partition["missing_answers"] == 113
assert int(observed.sum()) == 712827
assert np.all(Y[observed] < np.broadcast_to(option_mask.sum(axis=1), Y.shape)[observed])
assert np.all(observed.sum(axis=1) > 0)
features_t = torch.tensor(E, device=device)
mask_t = torch.tensor(option_mask, device=device)
targets_t = torch.tensor(Y, device=device)
print(f"Training: {len(person_ids):,} people, {len(training_questions)} questions, {observed.sum():,} observed answers")


Training: 1,658 people, 430 questions, 712,827 observed answers


## 3. Shared scorer + eight-number respondent vectors
`score = beta · answer_features + respondent_vectorᵀ W answer_features`.
Softmax compares valid options within a question. Optimize observed-answer cross-entropy plus a small L2 penalty. PCA and Qwen stay fixed.


In [4]:
class SharedResponseModel(nn.Module):
    def __init__(self, n_people, n_features=64, latent_dim=8):
        super().__init__()
        self.beta = nn.Parameter(torch.zeros(n_features))
        self.W = nn.Parameter(torch.randn(latent_dim, n_features) * 0.05)
        self.respondents = nn.Embedding(n_people, latent_dim)
        nn.init.normal_(self.respondents.weight, std=0.1)

    def forward(self, person_indices, answer_features, valid_options, use_person=True):
        base = answer_features @ self.beta
        if use_person:
            preferences = answer_features @ self.W.T
            logits = base + torch.einsum("bd,qkd->bqk", self.respondents(person_indices), preferences)
        else:
            logits = base.unsqueeze(0).expand(len(person_indices), -1, -1)
        return logits.masked_fill(~valid_options.unsqueeze(0), -1e9)

model = SharedResponseModel(len(person_ids), latent_dim=config["latent_dimension"]).to(device)
parameter_count = sum(p.numel() for p in model.parameters())
assert parameter_count == 64 + 8 * 64 + len(person_ids) * 8
assert not features_t.requires_grad
loss_fn = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])

@torch.no_grad()
def evaluate(use_person=True, scorer=None):
    scorer = model if scorer is None else scorer
    scorer.eval()
    total_loss, total_correct, total_observed = 0.0, 0, 0
    for start in range(0, len(person_ids), config["batch_size"]):
        ids = torch.arange(start, min(start + config["batch_size"], len(person_ids)), device=device)
        logits = scorer(ids, features_t, mask_t, use_person=use_person)
        targets = targets_t[ids]
        valid = targets >= 0
        count = int(valid.sum().item())
        loss = loss_fn(logits.reshape(-1, max_options), targets.reshape(-1))
        total_loss += float(loss.item()) * count
        total_correct += int(((logits.argmax(-1) == targets) & valid).sum().item())
        total_observed += count
    return {"nll": total_loss / total_observed, "accuracy": total_correct / total_observed}

initial = evaluate()
counts = np.broadcast_to(option_mask.sum(axis=1), Y.shape)[observed]
uniform = {"nll": float(np.log(counts).mean()), "expected_accuracy": float((1 / counts).mean())}
print(f"Trainable parameters: {parameter_count:,}; initial training NLL: {initial['nll']:.4f}")


Trainable parameters: 13,840; initial training NLL: 1.2960


## 4. Train for a fixed 100 epochs


In [5]:
rng = np.random.default_rng(config["seed"])
history = [{"epoch": 0, **initial, "elapsed_seconds": 0.0}]
start = time.monotonic()
for epoch in range(1, config["epochs"] + 1):
    model.train()
    order = rng.permutation(len(person_ids))
    for offset in range(0, len(order), config["batch_size"]):
        ids = torch.tensor(order[offset:offset + config["batch_size"]], dtype=torch.long, device=device)
        logits = model(ids, features_t, mask_t)
        nll = loss_fn(logits.reshape(-1, max_options), targets_t[ids].reshape(-1))
        penalty = model.beta.square().mean() + model.W.square().mean() + model.respondents(ids).square().mean()
        loss = nll + config["l2_penalty"] * penalty
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at epoch {epoch}")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config["gradient_clip_norm"], error_if_nonfinite=True)
        optimizer.step()
    if epoch == 1 or epoch % 10 == 0 or epoch == config["epochs"]:
        metrics = evaluate()
        elapsed = time.monotonic() - start
        history.append({"epoch": epoch, **metrics, "elapsed_seconds": elapsed})
        print(f"Epoch {epoch:3d}: training NLL {metrics['nll']:.4f}, accuracy {metrics['accuracy']:.1%}, elapsed {elapsed:.1f}s", flush=True)
training_seconds = time.monotonic() - start
final = evaluate()
shared_only = evaluate(use_person=False)
assert final["nll"] < initial["nll"] and final["nll"] < uniform["nll"]
assert final["nll"] < shared_only["nll"], "Respondent vectors did not improve training fit"
assert all(torch.isfinite(p).all().item() for p in model.parameters())
print(f"Finished in {training_seconds:.1f}s. These are training-fit metrics, not held-out performance.")


Epoch   1: training NLL 1.2590, accuracy 41.3%, elapsed 0.1s


Epoch  10: training NLL 1.1741, accuracy 47.9%, elapsed 0.7s


Epoch  20: training NLL 1.1425, accuracy 50.3%, elapsed 1.4s


Epoch  30: training NLL 1.1310, accuracy 51.0%, elapsed 2.1s


Epoch  40: training NLL 1.1273, accuracy 51.3%, elapsed 2.9s


...

Epoch  70: training NLL 1.1230, accuracy 51.5%, elapsed 5.2s


Epoch  80: training NLL 1.1224, accuracy 51.5%, elapsed 5.9s


Epoch  90: training NLL 1.1222, accuracy 51.6%, elapsed 6.5s


Epoch 100: training NLL 1.1219, accuracy 51.6%, elapsed 7.1s


Finished in 7.1s. These are training-fit metrics, not held-out performance.


## 5. Save and verify
Save PCA parameters, transformed features and model arrays with respondent IDs. Reload them and verify probabilities, option masking and missing-answer handling. The 1,658 background vectors are not the final 100 personas.


In [6]:
model.eval()
checkpoint_path = OUT / "shared_response_model.npz"
np.savez(checkpoint_path, beta=model.beta.detach().cpu().numpy(), W=model.W.detach().cpu().numpy(),
         respondent_vectors=model.respondents.weight.detach().cpu().numpy(), person_ids=np.array(person_ids),
         training_item_ids=np.array(training_questions))
with np.load(checkpoint_path, allow_pickle=False) as saved:
    assert saved["person_ids"].tolist() == person_ids
    assert saved["respondent_vectors"].shape == (1658, 8)
    reloaded = SharedResponseModel(len(person_ids), latent_dim=8).to(device)
    reloaded.load_state_dict({"beta": torch.tensor(saved["beta"], device=device),
                              "W": torch.tensor(saved["W"], device=device),
                              "respondents.weight": torch.tensor(saved["respondent_vectors"], device=device)})
    reloaded.eval()
    for a, b in zip(model.parameters(), reloaded.parameters()):
        assert torch.equal(a, b)

with np.load(pca_path, allow_pickle=False) as saved:
    reconstructed = ((embeddings - saved["mean"]) @ saved["components"].T) / np.sqrt(saved["explained_variance"])
np.testing.assert_allclose(np.load(feature_path, allow_pickle=False), reconstructed, atol=2e-5, rtol=2e-4)
with torch.no_grad():
    ids = torch.tensor([0, 1, len(person_ids) - 1], device=device)
    logits = reloaded(ids, features_t, mask_t)
    original = model(ids, features_t, mask_t)
    torch.testing.assert_close(logits, original, rtol=0, atol=0)
    # Independent NumPy calculation checks the intended scoring equation.
    person_preferences = model.respondents(ids).cpu().numpy() @ model.W.cpu().numpy() + model.beta.cpu().numpy()
    reference = np.einsum("pd,qkd->pqk", person_preferences, E)
    reference = np.where(option_mask[None, :, :], reference, -1e9)
    np.testing.assert_allclose(logits.cpu().numpy(), reference, atol=2e-5, rtol=2e-4)
    probabilities = logits.softmax(-1)
    assert torch.isfinite(probabilities).all().item()
    torch.testing.assert_close(probabilities.sum(-1), torch.ones_like(probabilities[..., 0]), atol=1e-6, rtol=1e-6)
    assert probabilities.masked_select(~mask_t.unsqueeze(0).expand_as(probabilities)).eq(0).all().item()
    # Missing labels contribute no loss.
    toy_logits = torch.tensor([[0.3, -0.2], [0.8, 0.1]], device=device)
    torch.testing.assert_close(nn.functional.cross_entropy(toy_logits, torch.tensor([0, -1], device=device), ignore_index=-1),
                               nn.functional.cross_entropy(toy_logits[:1], torch.tensor([0], device=device)))

for path, digest in input_hashes.items():
    assert sha256(ROOT / path) == digest
report = {
    "status": "passed", "config": config, "input_sha256": input_hashes,
    "device": str(device), "parameters": parameter_count,
    "pca": {"fit_option_count": len(training_rows), "fit_question_count": len(training_questions),
            "output_dimension": 64, "whiten": True, "training_variance_retained": float(pca.explained_variance_ratio_.sum()),
            "within_question_option_variance_retained": option_variance_retained,
            "seconds": pca_seconds},
    "training": {"respondents": len(person_ids), "observed_answers": int(observed.sum()),
                 "missing_answers_skipped": int((~observed).sum()), "seconds": training_seconds,
                 "initial": initial, "final": final, "uniform_baseline": uniform,
                 "shared_only_after_training": shared_only, "history": history},
    "checks": ["PCA fit exclusively on training options", "complete option-to-feature mapping",
               "only background respondents and permitted questions loaded", "missing answers ignored",
               "finite parameters and reduced training loss", "respondent vectors improve training fit", "model checkpoint round trip",
               "PCA transform round trip", "independent NumPy scoring equation", "probabilities sum to one", "padded options have zero probability",
               "input files unchanged"],
    "validation_or_test_answers_loaded": False, "development_or_final_evidence_loaded": False,
    "qwen_weights_updated": False, "final_personas_trained": False,
    "limitations": ["Initial fixed hyperparameters, not selected on validation performance.",
                    "Training accuracy is not an estimate of accuracy for new people or questions.",
                    "Eight latent coordinates are learned preferences, without fixed human-interpretable labels.",
                    "GPU reruns may differ slightly despite fixed seeds."],
    "versions": {p: version(p) for p in ["torch", "numpy", "scikit-learn"]},
    "files": {str(p.relative_to(ROOT)): {"sha256": sha256(p), "bytes": p.stat().st_size}
              for p in [pca_path, feature_path, checkpoint_path]},
    "next_step": "Freeze the shared scorer, fit development respondent vectors from their 92 evidence items, and evaluate validation questions."
}
report_path = OUT / "shared_response_training.json"
report_path.write_text(json.dumps(report, indent=2) + "\n")
display(pd.DataFrame([{"model": "initial", **initial}, {"model": "trained", **final},
                      {"model": "trained, respondent contribution removed", **shared_only}]))
print(f"Saved PCA, features, shared scorer, 1,658 respondent vectors and {report_path.relative_to(ROOT)}")


,model,nll,accuracy
0,initial,1.296011,0.308643
1,trained,1.121877,0.515817
2,"trained, respondent contribution removed",1.275465,0.399572


Saved PCA, features, shared scorer, 1,658 respondent vectors and outputs/shared_response_training.json
